<a href="https://colab.research.google.com/github/Xv-123/dl/blob/main/templates/09_causal_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/09_causal_attention.ipynb)

# 🔴 Hard: Causal Self-Attention

Implement **causal (masked) self-attention** — the attention used in GPT-style decoders.

Same as softmax attention, but each position can **only attend to itself and earlier positions** (no peeking at future tokens).

$$\text{scores}_{ij} = \begin{cases} \frac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### Signature
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- Position $i$ can only attend to positions $\le i$
- You **may** use `torch.softmax`, `torch.bmm`, `torch.triu`

In [11]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


^C


In [23]:
import torch
import math
from torch import nn

In [24]:
def sequence_mask(X, valid_len, value=0):
    """Mask irrelevant entries in sequences.

    Defined in :numref:`sec_utils`"""
    maxlen = X.size(1)
    mask = torch.arange((maxlen), dtype=torch.float32,
                        device=X.device)[None, :] < valid_len[:, None]
    X[~mask] = value
    return X

In [25]:
def masked_softmax(X,valid_lens):
    "通过在最后一个轴上遮蔽元素来执行softmax操作"
    if valid_lens is None:
        return nn.functional.softmax(X,dim=-1)
    else:
        shape = X.shape
        if valid_lens.dim() == 1:
            valid_lens = torch.repeat_interleave(valid_lens,shape[1])
        #展平成一维的，[1,2,3,4,1,2,3,4………………] 总共有batch*num_heads*num_step个数字
        else:
            valid_lens = valid_lens.reshape(-1)
        #X.reshape(-1,shape[-1]),X：batch*num_head*num_query, num_key_value 每一行表示一个query对应的分数
        X = sequence_mask(X.reshape(-1,shape[-1]),valid_lens,value=-1e6)
        return nn.functional.softmax(X.reshape(shape),dim=-1)


In [26]:
# ✏️ YOUR IMPLEMENTATION HERE
def causal_attention(Q, K, V):
    # 1. 缩放点积注意力
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

    # 2. 生成正确形状的 valid_lens = [[1,2,3,4]]
    # (batch_size, seq_len) 即 (1,4)
    batch_size, seq_len, _ = Q.shape
    valid_lens = torch.arange(1, seq_len + 1, device=Q.device)  # [1,2,3,4]
    valid_lens = valid_lens.unsqueeze(0).repeat(batch_size, 1)    # [[1,2,3,4]]，变成二维，第一维是batch_size

    # 3. 调用 masked_softmax
    attn_weights = masked_softmax(scores, valid_lens)

    # 4. 加权
    output = torch.matmul(attn_weights, V)
    return output

In [27]:
# 🧪 Debug
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)          # (1, 4, 8)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # should be True

Output shape: torch.Size([1, 4, 8])
Pos 0 == V[0]? True


In [28]:
from torch_judge import check
check('causal_attention')


🧪 Testing: Causal Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (16.0ms)
  ✅ [2/4] Future tokens don't affect past (5.0ms)
  ✅ [3/4] First position only sees itself (1.1ms)
  ✅ [4/4] Gradient flow (26.0ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (48.0ms total)
  Progress saved. Run status() to see your dashboard.

